In [1]:
import json
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

finetuned_model_path = "./merged_model"
test_data_path = "./intent_test.jsonl"

print(f"加载微调后的模型: {finetuned_model_path}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=finetuned_model_path,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=False,
    trust_remote_code=True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1015 17:49:45.825000 7644 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1015 17:49:45.856000 7644 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.13.0+cu130)
    Python  3.10.19 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more

Unsloth: xformers is installed but cannot load with torch 2.13.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
加载微调后的模型: ./merged_model
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.19.0. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: WARNING `trust_remote_code` is Tru

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [2]:
print(f"加载测试数据: {test_data_path}")
test_samples = []
with open(test_data_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        messages = data.get("messages", [])
        user_content = None
        assistant_content = None
        for msg in messages:
            if msg["role"] == "user":
                user_content = msg["content"]
            elif msg["role"] == "assistant":
                assistant_content = msg["content"]
        if user_content and assistant_content:
            test_samples.append({
                "input": user_content,
                "expected": assistant_content
            })

print(f"共加载 {len(test_samples)} 条测试样本")

加载测试数据: ./intent_test.jsonl
共加载 85 条测试样本


In [3]:
generated_responses = []
correct_count = 0

for i, sample in enumerate(test_samples):
    messages = [{"role": "user", "content": sample["input"]}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated_responses.append(response)

    gen_stripped = response
    exp_stripped = sample["expected"].strip()
    is_correct = gen_stripped == exp_stripped
    if is_correct:
        correct_count += 1

    status = "✅" if is_correct else "❌"
    print(f"\n{status} 第{i+1}条")
    if not is_correct:
        print(f"  生成: {gen_stripped[:200]}")
        print(f"  期望: {exp_stripped[:200]}")

accuracy = correct_count / len(test_samples) if len(test_samples) > 0 else 0.0
print(f"\n========== 评估结果 ==========")
print(f"总样本数: {len(test_samples)}")
print(f"完全匹配数: {correct_count}")
print(f"准确率: {accuracy:.2%}")

Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ 第1条


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ 第2条


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


✅ 第82条


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ 第83条


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ 第84条

✅ 第85条

========== 评估结果 ==========
总样本数: 85
完全匹配数: 78
准确率: 91.76%
